<a href="https://colab.research.google.com/github/parraavinaroberto-a11y/Sesion1/blob/main/Sesion3IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 3 — Ecosistema IA: HuggingFace, LangChain y Modelos

**Módulo: Python para IA** | Máster en Inteligencia Artificial

## Objetivos de esta sesión

- Explorar **HuggingFace**: Model Hub, Datasets, Pipeline API.
- Consumir **APIs de LLMs** desde Python (Gemini).
- Construir un **sistema RAG** con LangChain y Chroma.
- Entender **modelos locales** con Ollama (demo del profesor).

> 🧭 Hoy entramos en el ecosistema real de la IA aplicada: las herramientas y plataformas que usaréis en las asignaturas del Máster y en vuestros proyectos.

### ¿Qué cambia en esta sesión?

En las sesiones anteriores programamos "desde cero": funciones, clases, algoritmos. En esta sesión damos un salto conceptual importante: **dejamos de construir modelos** y empezamos a **usar modelos que otros ya han construido**.

Esto refleja la realidad de la industria actual: la gran mayoría de aplicaciones de IA no entrenan modelos desde cero. En su lugar, utilizan **modelos pre-entrenados** (como los disponibles en HuggingFace) o **APIs de LLMs** (como Gemini) y los adaptan a su caso de uso específico. Saber navegar este ecosistema es tan importante como saber programar.

Al final de esta sesión seréis capaces de:
1. Usar modelos pre-entrenados de HuggingFace para analizar texto, clasificar, generar y traducir.
2. Hacer llamadas programáticas a la API de Gemini con control total sobre el comportamiento del modelo.
3. Construir un sistema RAG que responda preguntas basándose en vuestros propios documentos.
4. Entender cuándo y por qué usar modelos locales frente a APIs cloud.

---
# Bloque 1 — HuggingFace: El Hub de la IA (20 min)

> 🎯 HuggingFace es la plataforma que ha democratizado el acceso a la IA. Antes, usar un modelo de NLP requería semanas de trabajo y conocimientos avanzados de Deep Learning. Hoy, con HuggingFace, puedes usar modelos state-of-the-art con **3 líneas de código**.

## ¿Qué es HuggingFace?

[HuggingFace](https://huggingface.co/) es el **GitHub de la IA**: una plataforma abierta que alberga modelos, datasets y aplicaciones de Machine Learning. Fundada en 2016, se ha convertido en el punto de referencia para compartir y usar modelos de IA. Casi todos los modelos importantes (Llama de Meta, Mistral, Stable Diffusion, BERT, GPT-2...) están disponibles en HuggingFace.

### Los tres pilares de HuggingFace

| Componente | Qué es | Números | Ejemplos |
|-----------|--------|---------|----------|
| **Model Hub** | Repositorio de modelos preentrenados | +800.000 modelos | GPT-2, BERT, Llama 3, Stable Diffusion, Whisper |
| **Datasets Hub** | Repositorio de datasets abiertos | +150.000 datasets | IMDB (opiniones), SQuAD (preguntas), MNIST (dígitos) |
| **Spaces** | Apps de demo desplegadas (Gradio/Streamlit) | +400.000 spaces | Chatbots, generadores de imágenes, demos interactivas |

### Conceptos clave que necesitas entender

- **Model Card**: cada modelo en HuggingFace tiene una documentación que describe qué hace, cómo se entrenó, sus limitaciones, métricas de rendimiento y ejemplos de uso. **Siempre lee la Model Card** antes de usar un modelo — te dirá si es adecuado para tu caso.

- **Pipeline**: es la API de alto nivel de HuggingFace. Abstrae toda la complejidad (descargar modelo, tokenizar texto, hacer inferencia, parsear resultados) en una sola función. Es lo que usaremos hoy.

- **Transformers**: la librería Python de HuggingFace para cargar y usar cualquier modelo del Hub. Los "transformers" son la arquitectura de red neuronal (inventada por Google en 2017) que está detrás de GPT, BERT, Llama y prácticamente todos los modelos de lenguaje modernos.

- **Tokenizer**: antes de que un modelo pueda procesar texto, necesita convertirlo en números. El tokenizer divide el texto en **tokens** (subpalabras) y les asigna un ID numérico. Cada modelo tiene su propio tokenizer.

- **Embeddings**: representaciones numéricas (vectores) de texto que capturan su significado semántico. Textos similares tienen embeddings similares. Los usaremos en RAG.

> 🎓 En el **workshop previo** ya habéis creado vuestra cuenta de HuggingFace. Si no lo habéis hecho, id a [huggingface.co/join](https://huggingface.co/join) — es gratuito.

In [1]:
# Instalar Transformers (ya viene en Colab)
# !pip install transformers -q

from transformers import pipeline

# Pipeline de Análisis de Sentimiento
clasificador = pipeline("sentiment-analysis")

textos = [
    "Me encanta este curso de IA, es muy práctico!",
    "El tiempo de espera fue terrible y la comida estaba fría",
    "El hotel estaba bien, nada especial",
    "Increíble experiencia, repetiré sin duda"
]

resultados = clasificador(textos)

for texto, resultado in zip(textos, resultados):
    emoji = "✅" if resultado["label"] == "POSITIVE" else "❌"
    print(f"{emoji} [{resultado['label']}: {resultado['score']:.3f}] {texto}")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

✅ [POSITIVE: 0.846] Me encanta este curso de IA, es muy práctico!
❌ [NEGATIVE: 0.877] El tiempo de espera fue terrible y la comida estaba fría
❌ [NEGATIVE: 0.879] El hotel estaba bien, nada especial
❌ [NEGATIVE: 0.989] Increíble experiencia, repetiré sin duda


In [2]:
from transformers import pipeline

# Especificamos un modelo más potente (RoBERTa)
# Este modelo devuelve 'positive', 'neutral', 'negative' en lugar de solo binario
model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"

clasificador_pro = pipeline("sentiment-analysis", model=model_path)

textos = [
    "Me encanta este curso de IA, es muy práctico!",
    "El tiempo de espera fue terrible y la comida estaba fría",
    "El hotel estaba bien, nada especial",
    "Increíble experiencia, repetiré sin duda"
]

resultados = clasificador_pro(textos)

print(f"Usando el modelo: {model_path}\n")
for texto, resultado in zip(textos, resultados):
    label = resultado['label'].upper()
    score = resultado['score']

    # Lógica de iconos mejorada
    emoji = "✅" if label == "POSITIVE" else "❌" if label == "NEGATIVE" else "😐"

    print(f"{emoji} [{label}: {score:.3f}] {texto}")

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Usando el modelo: cardiffnlp/twitter-roberta-base-sentiment-latest

✅ [POSITIVE: 0.682] Me encanta este curso de IA, es muy práctico!
❌ [NEGATIVE: 0.922] El tiempo de espera fue terrible y la comida estaba fría
😐 [NEUTRAL: 0.757] El hotel estaba bien, nada especial
😐 [NEUTRAL: 0.804] Increíble experiencia, repetiré sin duda


## Pipelines disponibles

La Pipeline API de HuggingFace es el punto de entrada más sencillo al ecosistema. Con una sola línea puedes cargar un modelo pre-entrenado y usarlo para una tarea concreta. HuggingFace detecta automáticamente el mejor modelo disponible para cada tarea (o puedes especificar uno concreto).

### Tareas de texto (NLP)

| Pipeline | Tarea | Ejemplo de uso |
|----------|-------|----------------|
| `sentiment-analysis` | Clasificación de sentimiento | "¿Esta reseña es positiva o negativa?" |
| `text-generation` | Completar/generar texto | Autocompletar, chatbots simples |
| `summarization` | Resumen automático | Condensar artículos largos |
| `question-answering` | Responder preguntas sobre un texto | Buscar información en documentos |
| `translation_en_to_es` | Traducción automática | Inglés → Español (y muchos otros pares) |
| `zero-shot-classification` | Clasificar sin entrenamiento previo | Categorizar textos en categorías arbitrarias |
| `ner` (Named Entity Recognition) | Detectar entidades (personas, lugares...) | Extraer nombres propios de un texto |
| `fill-mask` | Predecir palabras eliminadas | "Madrid es la [MASK] de España" → "capital" |

### Tareas multimodales

| Pipeline | Tarea | Ejemplo de uso |
|----------|-------|----------------|
| `image-to-text` | Describir imágenes | "¿Qué se ve en esta foto?" |
| `text-to-image` | Generar imágenes | Stable Diffusion |
| `automatic-speech-recognition` | Transcribir audio a texto | Whisper de OpenAI |
| `visual-question-answering` | Preguntas sobre imágenes | "¿De qué color es el coche?" |

### ¿Qué es "zero-shot"?

**Zero-shot** significa que el modelo puede realizar una tarea **sin haber sido entrenado específicamente** para ella. En la clasificación zero-shot, le das un texto y una lista de categorías, y el modelo decide cuál es más probable — sin necesidad de datos etiquetados. Esto es extremadamente útil para prototipar rápidamente.

> 💡 Vamos a probar varios de estos pipelines en las siguientes celdas. La primera ejecución tarda más porque descarga el modelo.

In [ ]:
# Zero-shot classification — clasificar sin entrenar
clasificador_zero = pipeline("zero-shot-classification")

texto = "Apple acaba de anunciar el nuevo iPhone con funciones de IA generativa"
categorias = ["tecnología", "deportes", "política", "entretenimiento", "ciencia"]

resultado = clasificador_zero(texto, candidate_labels=categorias)

print(f"Texto: {texto}")
for label, score in zip(resultado["labels"], resultado["scores"]):
    barra = "█" * int(score * 40)
    print(f"  {label:15s} {score:.3f} {barra}")

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Texto: Apple acaba de anunciar el nuevo iPhone con funciones de IA generativa
  tecnología      0.752 ██████████████████████████████
  ciencia         0.099 ███
  entretenimiento 0.096 ███
  deportes        0.027 █
  política        0.026 █


In [ ]:
# Generación de texto
generador = pipeline("text-generation", model="gpt2")

prompt = "Artificial Intelligence will transform education by"
resultados = generador(prompt, max_length=60, num_return_sequences=2, temperature=0.8)

for i, r in enumerate(resultados):
    print(f"--- Generación {i+1} ---")
    print(r["generated_text"])

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'temperature', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=60) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Generación 1 ---
Artificial Intelligence will transform education by providing a vast knowledge base, accessible to anyone willing to do some research and learning.

Not only that, and it will be an important step forward both for the government and the industry. The question is: where can we begin, and how long will it be?
--- Generación 2 ---
Artificial Intelligence will transform education by giving students the skills they need to make progress, but it will also help them avoid the pitfalls of traditional research that has lead them to distrust their own research.

The research is intended to provide better tools for students to assess and apply this new science in a wide range of areas, and it will help the future of AI in future generations.

"We have to think about how much of what we're trying to understand in the field is still underappreciated because it's quite difficult to get it validated," said Prof. Rolf Rönner from the University of Freiburg in Germany.

He said tha

In [ ]:
# Question Answering — responder preguntas sobre un contexto
from transformers import pipeline

qa = pipeline("question-answering")

contexto = """
La inteligencia artificial (IA) es una rama de la informática que busca crear
sistemas capaces de realizar tareas que normalmente requieren inteligencia humana.
Estas tareas incluyen el reconocimiento de voz, la toma de decisiones, la traducción
de idiomas y el reconocimiento visual. El machine learning es un subcampo de la IA
que permite a los sistemas aprender de los datos sin ser programados explícitamente.
El deep learning, a su vez, es un subcampo del machine learning que usa redes neuronales
profundas con múltiples capas para aprender representaciones complejas de los datos.
"""

preguntas = [
    "¿Qué es la inteligencia artificial?",
    "¿Qué es el machine learning?",
    "¿Qué usa el deep learning?"
]

for pregunta in preguntas:
    respuesta = qa(question=pregunta, context=contexto)
    print(f"❓ {pregunta}")
    print(f"💡 {respuesta['answer']} (confianza: {respuesta['score']:.3f})")

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5.
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

❓ ¿Qué es la inteligencia artificial?
💡 es una rama de la informática (confianza: 0.180)
❓ ¿Qué es el machine learning?
💡 un subcampo de la IA (confianza: 0.100)
❓ ¿Qué usa el deep learning?
💡 a su vez (confianza: 0.579)


### 🛠️ Ejercicio — Explora Pipelines

1. Ve a [huggingface.co/models](https://huggingface.co/models) y busca un modelo de **summarization**.
2. Usa `pipeline("summarization")` para resumir un texto largo (pega un artículo de Wikipedia).
3. Experimenta con los parámetros `max_length` y `min_length`.

Pide a Gemini:
> *"Genera código para usar el pipeline de summarization de HuggingFace para resumir un texto de al menos 500 palabras. Muestra el resumen con su longitud."*

In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Texto de ejemplo de al menos 500 palabras (hecho un poco más largo para asegurar)
texto_largo = """La Revolución Industrial, que comenzó en Gran Bretaña a finales del siglo XVIII, marcó un punto de inflexión en la historia de la humanidad. Fue un período de profundas transformaciones económicas, sociales y tecnológicas que alteraron radicalmente la forma de vida de las personas. Antes de la revolución, la producción se basaba principalmente en la artesanía y la agricultura, con herramientas manuales y métodos tradicionales. La invención de la máquina de vapor por James Watt fue un catalizador clave, ya que proporcionó una fuente de energía eficiente que podía aplicarse a una multitud de procesos industriales, desde la minería hasta la fabricación textil.

La introducción de nuevas máquinas, como el telar mecánico y la hiladora Jenny, permitió una producción masiva de bienes que antes eran manufacturados de forma lenta y costosa. Esto llevó al surgimiento de fábricas y, con ellas, a un nuevo sistema de producción capitalista. Las ciudades crecieron rápidamente a medida que la gente emigraba del campo en busca de trabajo en las nuevas industrias. Sin embargo, este crecimiento trajo consigo problemas sociales significativos, como la sobrepoblación, las condiciones de vida insalubres, la explotación laboral de mujeres y niños, y largas jornadas de trabajo con salarios bajos.

El transporte también experimentó una revolución con la invención del ferrocarril y los barcos de vapor, que facilitaron el movimiento de materias primas y productos terminados a gran escala, conectando mercados y expandiendo el comercio a nivel global. Esto no solo impulsó el crecimiento económico, sino que también cambió la geografía del poder y la política mundial. Las naciones con una industrialización avanzada ganaron una ventaja competitiva, lo que llevó a un aumento del colonialismo y la expansión imperialista en busca de recursos y mercados.

La Revolución Industrial no fue un evento único, sino un proceso continuo que se extendió a lo largo de décadas y se propagó por Europa, América del Norte y otras partes del mundo. Sus efectos fueron de largo alcance y duraderos, sentando las bases para la sociedad moderna. La producción en masa, la urbanización, el surgimiento de una clase trabajadora y una clase media, el desarrollo de nuevas ideologías políticas y económicas (como el socialismo y el capitalismo), y la aceleración del progreso tecnológico son todos legados directos de este período.

En las etapas posteriores, la Revolución Industrial vio el desarrollo de la electricidad, la química, el acero y el petróleo como nuevas fuerzas motrices. Estos avances llevaron a una segunda fase de industrialización, con la producción en cadena y la gestión científica del trabajo, popularizadas por figuras como Henry Ford. Los inventos como el automóvil, el teléfono y la bombilla eléctrica transformaron aún más la vida cotidiana y la infraestructura de las ciudades. La capacidad de generar y distribuir energía a gran escala permitió que la industria se deslocalizara y diversificara, llevando a la creación de nuevas industrias y a una mayor especialización del trabajo.

Los impactos ambientales de la Revolución Industrial también comenzaron a hacerse evidentes, con la contaminación del aire y el agua debido a las fábricas y la quema de combustibles fósiles. Aunque en ese momento no se comprendía completamente la magnitud de estos efectos, sentaron las bases para los desafíos ambientales que enfrentamos hoy. En resumen, la Revolución Industrial fue un fenómeno complejo y multifacético que redefinió la relación del ser humano con el trabajo, la tecnología y el medio ambiente, y cuyas reverberaciones todavía se sienten en el siglo XXI.

Otro aspecto crucial fue la transformación de la estructura social. La antigua jerarquía feudal y agraria fue reemplazada por una sociedad más estratificada, con la burguesía industrial y financiera en la cima y el proletariado (la clase obrera industrial) en la base. Esta nueva estructura dio lugar a movimientos obreros y a la lucha por los derechos laborales, que eventualmente conducirían a reformas sociales y a la creación de sindicatos. La educación también comenzó a expandirse, aunque lentamente al principio, para satisfacer la demanda de mano de obra cualificada y para inculcar la disciplina necesaria en el nuevo entorno fabril. La ciencia y la tecnología se integraron cada vez más en la producción, creando un ciclo de innovación que ha continuado hasta nuestros días.

En conclusión, la Revolución Industrial fue un período de cambio sin precedentes que sentó las bases para el mundo moderno. Sus innovaciones tecnológicas, su impacto en la organización social y económica, y sus consecuencias a largo plazo, tanto positivas como negativas, son fundamentales para comprender la trayectoria de la humanidad en los últimos dos siglos. Este período no solo cambió cómo se producían los bienes, sino también cómo vivía, trabajaba y se organizaba la sociedad, abriendo la puerta a la era de la información y la globalización que experimentamos hoy."""

print(f"Longitud del texto original: {len(texto_largo.split())} palabras\n")

# Cargar tokenizador y modelo para summarization (t5-base es un buen modelo para esto)
tokenizer = AutoTokenizer.from_pretrained("t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("t5-base")

# Preparar el texto de entrada para el modelo T5 (se requiere el prefijo "summarize: ")
input_text = "summarize: " + texto_largo
inputs = tokenizer(input_text, return_tensors="pt", max_length=1024, truncation=True)

# Generar el resumen
summary_ids = model.generate(
    inputs["input_ids"],
    max_length=150,
    min_length=50,
    num_beams=4,  # Usar beam search para mejor calidad
    early_stopping=True
)

resumen_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# Mostrar el resumen y su longitud
print("--- Resumen ---")
print(resumen_text)
print(f"\nLongitud del resumen: {len(resumen_text.split())} palabras")

Longitud del texto original: 784 palabras



spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

--- Resumen ---
la Revolución Industrial, que comenzó en Gran Bretaa a finales del siglo XVIII, marcó un punto de inflexión en la historia de la humanidad . la producción se basaba principalmente en la artesana y la agricultura, con herramientas manuales y métodos tradicionales .

Longitud del resumen: 43 palabras


---
# Bloque 2 — APIs de LLMs desde Python (20 min)

> 🎯 En el bloque anterior usamos modelos pre-entrenados de HuggingFace que se ejecutan en nuestra máquina (o en Colab). Ahora vamos a consumir modelos **mucho más grandes y potentes** (Gemini, GPT-4, Claude) que se ejecutan en servidores remotos y se acceden mediante **APIs**.

### Pipeline vs API: ¿cuándo usar cada una?

| Característica | Pipeline (HuggingFace) | API (Gemini, GPT...) |
|---------------|----------------------|----------------------|
| **Dónde se ejecuta** | En tu máquina / Colab | En servidores del proveedor |
| **Tamaño del modelo** | Pequeño-mediano (hasta ~7B) | Muy grandes (100B+) |
| **Calidad** | Buena para tareas específicas | Excelente para tareas generales |
| **Coste** | Gratis (solo tu hardware) | Gratis/de pago según proveedor |
| **Privacidad** | Total — datos no salen de tu máquina | Los datos viajan al servidor |
| **Personalización** | Puedes fine-tunar el modelo | Control limitado (prompt + parámetros) |
| **Latencia** | Depende de tu hardware | Depende de la red + cola del servidor |

En la industria real, **se usan ambos**: APIs para interacción general y generación (chatbots, asistentes), y modelos locales para tareas específicas donde la privacidad o latencia importan.

## Consumir LLMs por API

Los grandes modelos de lenguaje (GPT-4, Gemini, Claude) no se ejecutan en tu máquina — son demasiado grandes (cientos de miles de millones de parámetros). En su lugar, haces **llamadas HTTP** a servidores del proveedor (Google, OpenAI, Anthropic) que ejecutan el modelo y te devuelven la respuesta.

### Estructura estándar de una llamada a LLM

Casi todas las APIs de LLMs siguen el mismo patrón: envías una lista de **mensajes** con roles (`system`, `user`, `assistant`) y recibes una respuesta generada:

```python
respuesta = modelo.generar(
    messages=[
        {"role": "system", "content": "Eres un asistente experto en Python"},
        {"role": "user", "content": "Explica qué es un decorador"}
    ],
    temperature=0.7,      # Creatividad (0=determinista, 1=creativo)
    max_tokens=500         # Longitud máxima de respuesta
)
```

### Parámetros importantes

| Parámetro | Qué controla | Valores típicos |
|-----------|-------------|-----------------|
| `temperature` | Creatividad/aleatoriedad de la respuesta | 0.0 (determinista) → 1.0+ (creativo) |
| `max_tokens` | Longitud máxima de la respuesta | 100-4000 (depende del modelo) |
| `top_p` | Diversidad de vocabulario (nucleus sampling) | 0.1 (conservador) → 1.0 (diverso) |
| `system_instruction` | Comportamiento general del modelo | "Eres un asistente de Python..." |

> 💡 **temperature**: es el parámetro más importante. Para **código y datos** usa temperaturas bajas (0.0-0.3) → respuestas más precisas y consistentes. Para **texto creativo** usa temperaturas más altas (0.7-1.0) → respuestas más variadas y originales.

### Principales APIs disponibles

| Proveedor | Modelo | Gratis en Colab | Notas |
|-----------|--------|:---------------:|-------|
| **Google** | Gemini 2.0 Flash | ✅ Sí | Integrado en Colab, API gratuita generosa |
| OpenAI | GPT-4o | ❌ De pago | La más popular, referencia de la industria |
| Anthropic | Claude 3.5 | ❌ De pago | Excelente para código, razonamiento largo |
| Meta | Llama 3.1 | ✅ Local u HuggingFace | Open-source, ejecutable localmente |

> 🎯 **En este curso usaremos Gemini** porque es gratuito, tiene una API generosa y está integrado en Colab. Los conceptos que aprendáis son transferibles a cualquier otra API — todas siguen el mismo patrón.

In [6]:
# API de Gemini desde Colab (GRATUITA)
# En Colab, Gemini está disponible directamente

# Opción 1: Usar google.generativeai (disponible en Colab)
# !pip install google-generativeai -q

import google.generativeai as genai

# En Colab, puedes usar tu propia API key gratuita
# Ve a https://aistudio.google.com/apikey para obtenerla
# genai.configure(api_key="TU_API_KEY")

# O usa los secretos de Colab:
# from google.colab import userdata
# genai.configure(api_key=userdata.get('GEMINI_API_KEY'))

# Por ahora, si estás en Colab, prueba con:
try:
    from google.colab import userdata
    genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
    print("✅ API Key configurada desde secretos de Colab")
except Exception:
    print("⚠️ Para usar la API de Gemini:")
    print("   1. Ve a https://aistudio.google.com/apikey")
    print("   2. Crea una API Key gratuita")
    print("   3. En Colab: Menú lateral izquierdo → 🔑 Secrets → Añade GEMINI_API_KEY")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


⚠️ Para usar la API de Gemini:
   1. Ve a https://aistudio.google.com/apikey
   2. Crea una API Key gratuita
   3. En Colab: Menú lateral izquierdo → 🔑 Secrets → Añade GEMINI_API_KEY


In [7]:
# Llamada básica a Gemini
try:
    model = genai.GenerativeModel("gemini-2.5-flash")

    response = model.generate_content(
        "Explica qué es una red neuronal en 3 líneas, para alguien sin conocimientos técnicos."
    )

    print("🤖 Respuesta de Gemini:")
    print(response.text)
except Exception as e:
    print(f"⚠️ Error: {e}")
    print("Asegúrate de configurar la API Key (celda anterior)")

⚠️ Error: 
  No API_KEY or ADC found. Please either:
    - Set the `GOOGLE_API_KEY` environment variable.
    - Manually pass the key with `genai.configure(api_key=my_api_key)`.
    - Or set up Application Default Credentials, see https://ai.google.dev/gemini-api/docs/oauth for more information.
Asegúrate de configurar la API Key (celda anterior)


In [8]:
# Llamada con System Prompt y parámetros avanzados
try:
    model = genai.GenerativeModel(
        "gemini-2.5-flash",
        system_instruction="Eres un profesor de programación Python. "
                          "Explicas conceptos con ejemplos claros y analogías cotidianas. "
                          "Siempre incluyes un ejemplo de código funcional."
    )

    response = model.generate_content(
        "¿Qué es un decorador en Python?",
        generation_config=genai.GenerationConfig(
            temperature=0.3,       # Baja creatividad = más preciso
            max_output_tokens=500  # Máximo de tokens en la respuesta
        )
    )

    print("🎓 Respuesta con System Prompt:")
    print(response.text)
except Exception as e:
    print(f"⚠️ Error: {e}")

⚠️ Error: 
  No API_KEY or ADC found. Please either:
    - Set the `GOOGLE_API_KEY` environment variable.
    - Manually pass the key with `genai.configure(api_key=my_api_key)`.
    - Or set up Application Default Credentials, see https://ai.google.dev/gemini-api/docs/oauth for more information.


## Conversaciones Multi-turno (Chat)

Hasta ahora hemos hecho llamadas "sueltas" a la API — le mandamos un mensaje y recibimos una respuesta. Pero en la práctica, muchas aplicaciones necesitan **conversaciones con contexto**: el modelo debe recordar lo que se ha dicho antes para dar respuestas coherentes.

### ¿Cómo funciona la memoria en los LLMs?

Importante: los LLMs **no tienen memoria real**. Lo que hacemos es enviar **todo el historial de la conversación** en cada llamada. El modelo "recuerda" porque recibe los mensajes anteriores como contexto. Esto tiene dos implicaciones:

1. **Coste**: cada mensaje incluye todo el historial → a medida que la conversación crece, cada llamada cuesta más (más tokens).
2. **Límite**: cada modelo tiene un máximo de tokens que puede procesar (la "ventana de contexto"). Gemini 2.0 Flash tiene ~1M tokens, GPT-4o tiene ~128K.

### Gemini Chat API

La API de Gemini gestiona el historial automáticamente con `start_chat()`. Cada `send_message()` añade tu mensaje al historial, envía todo al modelo y añade la respuesta:

In [9]:
# Chat multi-turno con Gemini
try:
    model = genai.GenerativeModel("gemini-2.5-flash")
    chat = model.start_chat(history=[])

    # Turno 1
    r1 = chat.send_message("Hola! Estoy aprendiendo Python. ¿Qué es una lista?")
    print(f"🤖 Turno 1: {r1.text[:200]}...")

    # Turno 2 — el modelo recuerda el contexto
    r2 = chat.send_message("¿Y cómo la ordeno de mayor a menor?")
    print(f"🤖 Turno 2: {r2.text[:200]}...")

    # Turno 3 — sigue recordando
    r3 = chat.send_message("¿Hay alguna forma de hacerlo sin modificar la lista original?")
    print(f"🤖 Turno 3: {r3.text[:200]}...")

    print(f"📊 Historial del chat: {len(chat.history)} mensajes")
except Exception as e:
    print(f"⚠️ Error: {e}")

⚠️ Error: 
  No API_KEY or ADC found. Please either:
    - Set the `GOOGLE_API_KEY` environment variable.
    - Manually pass the key with `genai.configure(api_key=my_api_key)`.
    - Or set up Application Default Credentials, see https://ai.google.dev/gemini-api/docs/oauth for more information.


### API Keys — Buenas Prácticas de Seguridad

Las API Keys son como **contraseñas** que identifican tu cuenta ante el proveedor. Si alguien obtiene tu API key, puede hacer llamadas a la API en tu nombre (y potencialmente generar costes). Por eso es **crítico** protegerlas:

```python
# ❌ NUNCA hagas esto — la key queda expuesta en el código
genai.configure(api_key="AIzaSyAbCdEfGhIjKlMnOpQrStUvWxYz")

# ✅ BIEN — usar variables de entorno (scripts locales)
import os
genai.configure(api_key=os.environ["GEMINI_API_KEY"])

# ✅ BIEN — usar secretos de Colab (notebooks)
from google.colab import userdata
genai.configure(api_key=userdata.get("GEMINI_API_KEY"))

# ✅ BIEN — usar archivos .env (proyectos locales)
# pip install python-dotenv
from dotenv import load_dotenv
load_dotenv()  # Carga variables desde .env
genai.configure(api_key=os.environ["GEMINI_API_KEY"])
```

### Errores comunes con API Keys

| Error | Consecuencia | Cómo evitarlo |
|-------|-------------|---------------|
| Poner la key en el código | Si subes a GitHub, queda expuesta públicamente | Usar variables de entorno o secretos |
| Subir `.env` a Git | La key se filtra en el repositorio | Añadir `.env` al `.gitignore` |
| Compartir un notebook con la key | El receptor accede a tu cuenta | Limpiar outputs y usar secretos de Colab |

> ⚠️ **Si filtras una key**: ve inmediatamente a la consola del proveedor y **revócala**. Genera una nueva. Los bots escanean GitHub constantemente buscando API keys expuestas.

---
# Bloque 3 — LangChain y RAG (25 min)

> 🎯 Este es probablemente el bloque más relevante para la industria actual. **RAG** (Retrieval-Augmented Generation) es el patrón de arquitectura más utilizado en aplicaciones de IA generativa en producción: chatbots de soporte técnico, asistentes legales, buscadores empresariales, herramientas de análisis de documentación... todos usan RAG.

El concepto es sencillo: en vez de confiar ciegamente en lo que el LLM "sabe" (que puede estar desactualizado o ser incorrecto), le **proporcionamos documentos relevantes** antes de que responda. Así, la respuesta se basa en **información verificable** en lugar de en la "memoria" del modelo.

## ¿Qué es LangChain?

**LangChain** es un framework de Python que simplifica la construcción de aplicaciones complejas con LLMs. Si la API de Gemini te da acceso al modelo, LangChain te da las **"piezas de LEGO"** para construir aplicaciones completas sobre él.

### ¿Qué te permite hacer LangChain?

- 🔗 **Encadenar** operaciones: prompt → LLM → parser → acción → siguiente prompt (las "chains" que le dan nombre).
- 🧠 **Añadir memoria** a conversaciones: que el chatbot recuerde lo que dijiste hace 10 mensajes.
- 🔧 **Conectar herramientas** externas: búsqueda web, bases de datos SQL, calculadoras, APIs de terceros.
- 📚 **RAG**: buscar información en tus documentos y usarla para fundamentar las respuestas del LLM.
- 🤖 **Agentes**: LLMs que deciden qué herramienta usar según la pregunta del usuario.

### ¿Qué es RAG y por qué es tan importante?

**RAG** (Retrieval-Augmented Generation) resuelve tres problemas fundamentales de los LLMs:

1. **Corte de conocimiento**: los LLMs se entrenan con datos hasta cierta fecha. No saben nada posterior → RAG les da acceso a información actual.
2. **Alucinaciones**: los LLMs pueden inventar datos con total confianza → RAG los fundamenta en documentos reales, reduciendo las alucinaciones.
3. **Conocimiento privado**: los LLMs no conocen los documentos internos de tu empresa → RAG les "inyecta" esa información.

### Arquitectura de un sistema RAG

```
┌──────────────┐                ┌──────────────┐
│  Pregunta    │──────────────▶│  Embeddings  │
│  del usuario │                │  (vectorizar)│
└──────────────┘                └──────┬───────┘
                                       │ Vector de la pregunta
                                       ▼
                               ┌──────────────┐
                               │  Base de     │
                               │  datos       │◀── Documentos indexados
                               │  vectorial   │    (previamente vectorizados)
                               │  (Chroma)    │
                               └──────┬───────┘
                                       │ Top-K fragmentos más relevantes
                                       ▼
                               ┌──────────────┐
                               │     LLM      │
                               │  (Gemini)    │──▶ Respuesta fundamentada
                               │  + contexto  │    en tus documentos
                               └──────────────┘
```

### ¿Qué son los embeddings y la base de datos vectorial?

- **Embeddings**: son vectores numéricos (arrays de ~768-1536 números) que representan el **significado semántico** del texto. Textos similares en significado tendrán vectores cercanos en el espacio. "Madrid es la capital de España" y "La capital española es Madrid" tendrán embeddings muy similares, aunque las palabras sean diferentes.

- **Base de datos vectorial** (Chroma, Pinecone, Weaviate, FAISS...): almacena los embeddings de tus documentos y permite hacer **búsquedas por similitud** — dada una pregunta, encuentra los documentos cuyo significado es más parecido. Esto es mucho más potente que una búsqueda por palabras clave.

> 💡 En otras asignaturas del Máster (NLP, ML Predictivo) profundizaréis en cómo funcionan los embeddings matemáticamente. Aquí nos centramos en **usarlos**.

In [10]:
# Instalar dependencias para RAG
!pip install langchain langchain-google-genai langchain-community chromadb langchain-text-splitters -q

print("✅ Dependencias instaladas")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 103.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 106.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 399.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.

In [12]:
# Preparar documentos de ejemplo
# Imaginemos que estos son fragmentos de la documentación de tu empresa

documentos = [
    "La empresa TechCorp fue fundada en 2015 por María García en Madrid. "
    "Se especializa en soluciones de inteligencia artificial para el sector salud.",

    "TechCorp tiene actualmente 150 empleados distribuidos en tres oficinas: "
    "Madrid (sede central), Barcelona y Lisboa. La facturación en 2023 fue de 12 millones de euros.",

    "El producto principal de TechCorp es MediAI, una plataforma de diagnóstico "
    "asistido por IA que analiza imágenes médicas. Tiene certificación CE para uso clínico.",

    "TechCorp utiliza modelos de deep learning basados en arquitecturas ResNet y "
    "Vision Transformer (ViT) para el análisis de imágenes médicas. "
    "Los modelos se entrenan con datasets anonimizados de hospitales asociados.",

    "En 2024, TechCorp lanzó su nueva línea de producto: PharmaAI, un sistema de "
    "descubrimiento de fármacos asistido por IA que utiliza modelos generativos "
    "para proponer nuevas estructuras moleculares.",

    "La política de privacidad de TechCorp establece que todos los datos médicos "
    "se procesan en servidores europeos cumpliendo con GDPR. Los datos se anonimizan "
    "antes del entrenamiento y no se comparten con terceros.",

    "TechCorp colabora con el Hospital Universitario La Paz, el Hospital Clínic de "
    "Barcelona y el Hospital de Santa María de Lisboa para la validación clínica de MediAI.",

    "El equipo de investigación de TechCorp ha publicado 15 papers en conferencias "
    "top como NeurIPS, ICML y MICCAI. Su paper más citado trata sobre detección "
    "temprana de cáncer de mama con IA, con más de 500 citas."
]

print(f"📄 {len(documentos)} documentos preparados")
for i, doc in enumerate(documentos):
    print(f"   Doc {i+1}: {doc[:80]}...")

📄 8 documentos preparados
   Doc 1: La empresa TechCorp fue fundada en 2015 por María García en Madrid. Se especiali...
   Doc 2: TechCorp tiene actualmente 150 empleados distribuidos en tres oficinas: Madrid (...
   Doc 3: El producto principal de TechCorp es MediAI, una plataforma de diagnóstico asist...
   Doc 4: TechCorp utiliza modelos de deep learning basados en arquitecturas ResNet y Visi...
   Doc 5: En 2024, TechCorp lanzó su nueva línea de producto: PharmaAI, un sistema de desc...
   Doc 6: La política de privacidad de TechCorp establece que todos los datos médicos se p...
   Doc 7: TechCorp colabora con el Hospital Universitario La Paz, el Hospital Clínic de Ba...
   Doc 8: El equipo de investigación de TechCorp ha publicado 15 papers en conferencias to...


In [13]:
import google.generativeai as genai
from langchain_core.embeddings import Embeddings
from langchain_community.vectorstores import Chroma
from google.colab import userdata
import os

# Configuración de API
os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

class StableGoogleEmbeddings(Embeddings):
    def __init__(self, model_name: str):
        self.model_name = model_name

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [self.embed_query(text) for text in texts]

    def embed_query(self, text: str) -> list[float]:
        result = genai.embed_content(
            model=self.model_name,
            content=text,
            task_type="retrieval_document"
        )
        return result['embedding']

# Usamos el modelo confirmado en el paso de diagnóstico
embeddings = StableGoogleEmbeddings(model_name="models/gemini-embedding-001")

# 2. Crear base de datos vectorial
try:
    vectorstore = Chroma.from_texts(
        texts=documentos,
        embedding=embeddings,
        collection_name="techcorp_docs_final"
    )
    print(f"✅ Base de datos vectorial creada con {vectorstore._collection.count()} documentos")
except Exception as e:
    print(f"❌ Error: {e}")

SecretNotFoundError: Secret GEMINI_API_KEY does not exist.

### Intento de recuperación del Vector Store
Si el error persiste, asegúrate de que no haya habido un fallo de red durante la instalación superior.

In [15]:
# 3. Buscar documentos relevantes (retrieval)
# Esto es la "R" de RAG

pregunta = "¿Cuál es el producto principal de TechCorp?"

# Buscar los 3 documentos más relevantes
docs_relevantes = vectorstore.similarity_search(pregunta, k=3)

print(f"❓ Pregunta: {pregunta}")
print("📚 Documentos relevantes encontrados:")
for i, doc in enumerate(docs_relevantes):
    print(f"--- Documento {i+1} ---")
    print(doc.page_content)

NameError: name 'vectorstore' is not defined

In [16]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Configurar el LLM forzando la API v1 estable y usando el modelo solicitado
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=userdata.get("GEMINI_API_KEY"),
    temperature=0.2,
    version="v1"
)

# 2. Definir el prompt para RAG
template = """Responde la pregunta basándote únicamente en el siguiente contexto:
{context}

Pregunta: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

# 3. Crear la cadena RAG usando LCEL
rag_chain = (
    {"context": vectorstore.as_retriever(search_kwargs={"k": 3}), "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 4. Ejecutar preguntas
preguntas = [
    "¿Quién fundó TechCorp y cuándo?",
    "¿Cuántos empleados tiene y dónde están?",
    "¿Qué tecnologías usa MediAI para analizar imágenes?",
    "¿Cómo gestiona TechCorp la privacidad de los datos médicos?",
    "¿Cuál es el nuevo producto lanzado en 2024?"
]

for pregunta in preguntas:
    print(f"❓ {pregunta}")
    try:
        respuesta = rag_chain.invoke(pregunta)
        print(f"💡 {respuesta}\n")
    except Exception as e:
        print(f"❌ Error al generar respuesta: {e}\n")

SecretNotFoundError: Secret GEMINI_API_KEY does not exist.

### ¿Cómo funciona internamente el RAG paso a paso?

Cuando ejecutáis `qa_chain.invoke({"query": pregunta})`, internamente ocurren 5 pasos:

1. **Embedding de la pregunta**: tu pregunta se convierte en un vector numérico usando el mismo modelo de embeddings que se usó para indexar los documentos (en nuestro caso, `embedding-001` de Google).

2. **Búsqueda vectorial (Retrieval)**: el vector de la pregunta se compara con los vectores de todos los documentos almacenados en Chroma, usando **distancia coseno** como medida de similitud. Los documentos cuyo vector es más "cercano" al de la pregunta son los más relevantes.

3. **Selección de Top-K**: se seleccionan los K documentos más relevantes (en nuestro caso `k=3`). Demasiados llenan el contexto del LLM con información irrelevante; muy pocos pueden omitir datos importantes.

4. **Prompt aumentado (Augmented)**: LangChain construye automáticamente un prompt que incluye tanto los documentos recuperados como la pregunta del usuario: *"Usando la siguiente información: [doc1, doc2, doc3], responde a: [pregunta]"*.

5. **Generación**: el LLM (Gemini) genera la respuesta basándose **exclusivamente** en los documentos proporcionados, no en su conocimiento general. Esto reduce drásticamente las alucinaciones.

### ¿Dónde se usa RAG en la industria?

| Caso de uso | Documentos | Ejemplo |
|-------------|-----------|---------|
| **Soporte técnico** | Documentación de producto, FAQs | Chatbot que responde preguntas sobre tu software |
| **Legal** | Contratos, leyes, jurisprudencia | Asistente que busca precedentes legales |
| **Médico** | Papers científicos, guías clínicas | Herramienta de consulta para profesionales sanitarios |
| **Educación** | Apuntes, libros de texto, temarios | Tutor que responde basándose en el material del curso |
| **Empresarial** | Wikis internas, políticas, procesos | Asistente de onboarding para nuevos empleados |

> 💡 RAG es probablemente el patrón de IA más **demandado** profesionalmente en este momento. Si tenéis que quedaron con una sola cosa de esta sesión, que sea RAG.

### 🛠️ Ejercicio — Construye tu propio RAG

1. Copia los documentos de arriba y **cámbialos por información sobre un tema que te interese** (tu CV, un artículo, reglas de un juego, etc.)
2. Crea una base de datos vectorial con esos documentos.
3. Haz al menos 3 preguntas y comprueba que las respuestas se fundamentan en tus documentos.

> Pide a Gemini que te ayude: *"Genera 8 párrafos sobre [tu tema] para usarlos como documentos en un sistema RAG"*

In [18]:
# Pega aquí tu RAG personalizado

import google.generativeai as genai
from langchain_core.embeddings import Embeddings
from langchain_community.vectorstores import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from google.colab import userdata
import os

# Ensure chromadb is installed
!pip install chromadb -q

# --- 1. Documentos sobre Exploración Espacial ---
my_documents = [
    "La exploración espacial es el descubrimiento y exploración de cuerpos celestes y del espacio exterior, realizado por tecnologías espaciales en constante evolución. Ha sido una ambición humana desde tiempos inmemoriales, impulsada por la curiosidad científica y la búsqueda de conocimiento.",
    "La carrera espacial entre Estados Unidos y la Unión Soviética durante la Guerra Fría marcó un hito en la exploración espacial. Incluyó logros como el lanzamiento del Sputnik 1 en 1957 y la llegada del primer hombre a la Luna, Neil Armstrong, en 1969.",
    "Los telescopios espaciales como el Hubble y, más recientemente, el James Webb, han revolucionado nuestra comprensión del universo. Nos han proporcionado imágenes impresionantes y datos cruciales sobre galaxias lejanas, nebulosas y la formación estelar.",
    "Actualmente, la exploración de Marte es uno de los objetivos principales de varias agencias espaciales. Misiones como las de los rovers Curiosity y Perseverance de la NASA buscan signos de vida pasada y estudian la geología y la atmósfera marciana para futuras misiones tripuladas.",
    "La Estación Espacial Internacional (ISS) es un laboratorio orbital donde astronautas de múltiples países realizan experimentos científicos en microgravedad, vitales para el avance de la medicina, la física de materiales y la preparación para viajes de larga duración.",
    "El desarrollo de cohetes reutilizables por empresas como SpaceX ha reducido significativamente los costos de lanzamiento, abriendo la puerta a una nueva era de acceso al espacio y promoviendo la comercialización de la órbita baja terrestre.",
    "La búsqueda de exoplanetas, planetas fuera de nuestro sistema solar, es un campo de investigación vibrante. Se han descubierto miles de ellos, y la atención se centra en encontrar aquellos que podrían albergar vida.",
    "Las futuras misiones de exploración espacial incluyen el retorno a la Luna con el programa Artemis, el envío de sondas a lunas de Júpiter y Saturno (como Europa y Titán) para buscar océanos subterráneos, y la planificación de misiones tripuladas a Marte para la década de 2030."
]

print(f"📄 {len(my_documents)} documentos sobre exploración espacial preparados.")
for i, doc in enumerate(my_documents):
    print(f"   Doc {i+1}: {doc[:80]}...")

# --- 2. Configuración de API y Embeddings ---

api_key = None # Initialize api_key
vectorstore = None # Initialize vectorstore

try:
    # Intenta obtener la API key de los secretos de Colab
    api_key = userdata.get("GEMINI_API_KEY")
    if api_key:
        os.environ["GOOGLE_API_KEY"] = api_key
        genai.configure(api_key=api_key)
        print("✅ API Key de Gemini configurada desde secretos de Colab.")
    else:
        raise ValueError("GEMINI_API_KEY not found in Colab secrets.")
except Exception as e:
    print(f"⚠️ Error al configurar la API Key de Gemini: {e}")
    print("   Asegúrate de haber guardado tu API Key de Gemini en Colab Secrets como 'GEMINI_API_KEY'.")
    print("   El RAG no funcionará sin la API Key.")


# Definición de la clase StableGoogleEmbeddings para compatibilidad
class StableGoogleEmbeddings(Embeddings):
    def __init__(self, model_name: str):
        self.model_name = model_name

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        # Para evitar problemas con el límite de peticiones, se embedden uno por uno
        return [self.embed_query(text) for text in texts]

    def embed_query(self, text: str) -> list[float]:
        # Asegura que la API Key esté configurada antes de embedder
        if not os.environ.get("GOOGLE_API_KEY"): # Check os.environ as genai uses it internally
            raise ValueError("GOOGLE_API_KEY no configurada. Por favor, configura la API Key de Gemini.")
        result = genai.embed_content(
            model=self.model_name,
            content=text,
            task_type="retrieval_document"
        )
        return result['embedding']

# Usamos el modelo de embeddings adecuado
embeddings = StableGoogleEmbeddings(model_name="models/embedding-001")

# --- 3. Crear base de datos vectorial (Chroma) ---
if api_key:
    try:
        vectorstore = Chroma.from_texts(
            texts=my_documents,
            embedding=embeddings,
            collection_name="space_exploration_docs"
        )
        print(f"✅ Base de datos vectorial creada con {vectorstore._collection.count()} documentos.")
    except Exception as e:
        print(f"❌ Error al crear la base de datos vectorial: {e}")
else:
    print("Skipping vectorstore creation as API Key is not configured.")


# --- 4. Configurar el LLM para RAG ---
if api_key and vectorstore:
    llm = ChatGoogleGenerativeAI(
        model="gemini-pro", # Utiliza un modelo adecuado, como gemini-pro o gemini-1.5-flash
        google_api_key=api_key, # Pasa la API Key explícitamente
        temperature=0.2,
        version="v1"
    )

    # --- 5. Definir el prompt para RAG ---
    template = """Responde la pregunta basándote únicamente en el siguiente contexto:
{context}

Pregunta: {question}
"""
    prompt = ChatPromptTemplate.from_template(template)

    # --- 6. Crear la cadena RAG usando LCEL ---
    rag_chain = (
        {"context": vectorstore.as_retriever(search_kwargs={"k": 3}), "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    # --- 7. Ejecutar preguntas ---
    my_questions = [
        "¿Cuál fue un logro importante de la carrera espacial?",
        "¿Qué tipo de misiones se están realizando en Marte?",
        "¿Cómo ha impactado SpaceX en la exploración espacial?",
        "¿Qué son los telescopios espaciales Hubble y James Webb?"
    ]

    print("\n--- Realizando preguntas al sistema RAG ---")
    for pregunta in my_questions:
        print(f"❓ {pregunta}")
        try:
            respuesta = rag_chain.invoke(pregunta)
            print(f"💡 {respuesta}\n")
        except Exception as e:
            print(f"❌ Error al generar respuesta para '{pregunta}': {e}\n")
elif not api_key:
    print("Skipping LLM configuration and RAG chain creation as API Key is not configured.")
elif not vectorstore:
    print("Skipping LLM configuration and RAG chain creation as vectorstore was not initialized.")

📄 8 documentos sobre exploración espacial preparados.
   Doc 1: La exploración espacial es el descubrimiento y exploración de cuerpos celestes y...
   Doc 2: La carrera espacial entre Estados Unidos y la Unión Soviética durante la Guerra ...
   Doc 3: Los telescopios espaciales como el Hubble y, más recientemente, el James Webb, h...
   Doc 4: Actualmente, la exploración de Marte es uno de los objetivos principales de vari...
   Doc 5: La Estación Espacial Internacional (ISS) es un laboratorio orbital donde astrona...
   Doc 6: El desarrollo de cohetes reutilizables por empresas como SpaceX ha reducido sign...
   Doc 7: La búsqueda de exoplanetas, planetas fuera de nuestro sistema solar, es un campo...
   Doc 8: Las futuras misiones de exploración espacial incluyen el retorno a la Luna con e...
⚠️ Error al configurar la API Key de Gemini: Secret GEMINI_API_KEY does not exist.
   Asegúrate de haber guardado tu API Key de Gemini en Colab Secrets como 'GEMINI_API_KEY'.
   El RAG no funcio

---
# Bloque 4 — Modelos Locales con Ollama (15 min)

> 🎬 **Esta sección es una demo del profesor**. No necesitáis instalar Ollama para seguir la clase.

### ¿Por qué hablar de modelos locales?

Hasta ahora hemos usado dos formas de acceder a modelos: **HuggingFace Pipeline** (modelos pequeños que se ejecutan en Colab) y **APIs cloud** (modelos enormes en servidores de Google). Los **modelos locales** son una tercera opción cada vez más relevante: modelos de tamaño medio que se ejecutan directamente en tu ordenador, sin conexión a internet y sin enviar datos a ningún servidor.

## ¿Qué es Ollama?

[Ollama](https://ollama.com/) es una herramienta de línea de comandos que permite **descargar y ejecutar modelos de IA localmente** con un solo comando. Es como un "Docker para modelos de IA" — gestiona la descarga, la configuración y la ejecución de modelos sin que tengas que preocuparte por dependencias de CUDA, cuantización o conversión de formatos.

### ¿Cuándo usar modelos locales vs API cloud?

| Aspecto | API Cloud (Gemini, GPT) | Modelo Local (Ollama) |
|---------|:-----------------------:|:--------------------:|
| **Privacidad** | Datos viajan a servidores externos | Todo queda en tu máquina — ideal para datos sensibles |
| **Coste** | Gratis o pago por uso | Sin coste recurrente (solo tu hardware) |
| **Latencia** | Depende de internet + cola del servidor | Sin latencia de red — respuesta inmediata |
| **Calidad** | Modelos enormes (100B+ parámetros) | Modelos más pequeños (3B-70B) — menor calidad general |
| **Setup** | Solo necesitas una API key | Requiere GPU / RAM suficiente |
| **Offline** | ❌ Necesita internet siempre | ✅ Funciona completamente sin conexión |
| **Personalización** | Limitada a prompt/parámetros | Puedes fine-tunar, crear model files custom |

### ¿Cuándo tiene sentido usar Ollama?

- Trabajas con **datos confidenciales** (médicos, financieros, legales) que no puedes enviar a servers externos.
- Necesitas **baja latencia** constante (aplicaciones de tiempo real).
- Quieres **experimentar** con diferentes modelos sin gastar dinero en APIs.
- Vas a hacer **fine-tuning** o personalización profunda del modelo.
- Necesitas funcionar **offline** (en un avión, zona sin cobertura, o por política de empresa).

### Modelos populares en Ollama

| Modelo | Parámetros | RAM necesaria | Ideal para |
|--------|-----------|:-------------:|-----------|
| Phi-3 Mini | 3.8B | ~4 GB | Chat ligero, dispositivos con poca RAM |
| Gemma 2 | 9B | ~8 GB | Modelo de Google, buen equilibrio |
| Llama 3.1 | 8B | ~8 GB | Propósito general, muy buena calidad para su tamaño |
| Mistral | 7B | ~8 GB | Razonamiento, tareas analíticas |
| CodeLlama | 7B-34B | 8-32 GB | Especializado en generación de código |
| Llama 3.1 | 70B | ~48 GB | Calidad cercana a GPT-4 (requiere GPU potente) |

> 📏 **Regla práctica**: un modelo de ~7B parámetros necesita aproximadamente 8 GB de RAM y funciona razonablemente bien en un portátil moderno. A partir de 70B necesitas GPUs dedicadas.

## Demo en Vivo — Ollama

El profesor mostrará cómo instalar, configurar y usar Ollama en tiempo real. Tomad nota de los conceptos, pero **no necesitáis instalarlo** para seguir el curso.

### 1. Instalación y uso básico

Ollama se instala con un solo comando y funciona en macOS, Linux y Windows:

```bash
# Instalar Ollama (macOS/Linux)
curl -fsSL https://ollama.com/install.sh | sh

# Descargar un modelo (solo la primera vez, después queda en caché)
ollama pull llama3.1

# Chat interactivo en terminal
ollama run llama3.1
>>> ¿Qué es una red neuronal? Explícalo para alguien sin conocimientos técnicos.
```

La primera descarga lleva unos minutos (el modelo de 8B ocupa ~4.7 GB). Las siguientes ejecuciones son instantáneas.

### 2. Uso desde Python (API REST local)

Ollama expone una API REST en `localhost:11434` que puedes consumir desde cualquier lenguaje. Con Python y `requests`:

```python
import requests

response = requests.post("http://localhost:11434/api/generate", json={
    "model": "llama3.1",
    "prompt": "Explica qué es RAG en 3 líneas",
    "stream": False
})

print(response.json()["response"])
```

### 3. Integración con LangChain

Lo más potente: Ollama se integra directamente con LangChain, lo que significa que puedes **reemplazar la API de Gemini por un modelo local** cambiando solo una línea de código:

```python
from langchain_community.llms import Ollama

# Antes: llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", ...)
# Ahora:
llm = Ollama(model="llama3.1")
respuesta = llm.invoke("¿Qué ventajas tiene ejecutar un modelo localmente?")
print(respuesta)
```

Esto es una de las ventajas de usar LangChain como capa de abstracción: puedes **cambiar de proveedor** (Google → local → OpenAI) sin reescribir tu aplicación.

> 📝 **Si queréis probarlo en casa**: id a [ollama.com](https://ollama.com), descargad la app, y ejecutad `ollama run llama3.1`. Requiere al menos 8 GB de RAM. En Mac con Apple Silicon (M1/M2/M3) funciona especialmente bien.

---
# 🎮 Kahoot — Repaso de la Sesión 3

¡Vamos a repasar lo aprendido!

**Temas cubiertos:**
- HuggingFace: Hub, Pipeline, Transformers
- APIs de LLMs: Gemini, estructura de llamadas, API keys
- RAG: embeddings, bases vectoriales, LangChain
- Modelos locales: Ollama

> El link del Kahoot se compartirá durante la clase en directo.

---
## 📚 Recursos adicionales

- [HuggingFace — Documentación](https://huggingface.co/docs)
- [HuggingFace — Model Hub](https://huggingface.co/models)
- [Google AI Studio — API Key gratuita](https://aistudio.google.com/apikey)
- [LangChain — Documentación](https://python.langchain.com/docs/get_started/introduction)
- [Chroma — Base de datos vectorial](https://docs.trychroma.com/)
- [Ollama — Sitio oficial](https://ollama.com/)
- [RAG — Explicación de Anthropic](https://docs.anthropic.com/en/docs/build-with-claude/retrieval-augmented-generation)

## ⏭️ Próxima sesión

**Sesión 4: Interfaces, Deployment y Proyecto Integrador**
- Gradio: crear interfaces web para modelos de IA
- Streamlit: crear apps de datos e IA
- Deployment: HuggingFace Spaces, Streamlit Cloud
- Proyecto final integrador